# Акустическая атака по сторонним каналам на клавиатуру
## Учебный проект по кибербезопасности — Google Colab (T4 GPU)

Нейросеть обучается классифицировать нажатия клавиш по акустическим сигнатурам.
Каждое нажатие на механической клавиатуре даёт два транзиента: звук нажатия и звук отпускания.
Модель различает клавиши по mel-спектрограммам.

**Структура датасета:** `key.m4a`, `key1.m4a` на каждую клавишу  
**Тестовый файл:** `zbmpzbmp.m4a` — ожидаемый вывод: `z b m p z b m p`  

> Только в образовательных целях. Демонстрирует известный вектор атаки по сторонним каналам.


In [ ]:
# Установка зависимостей
!pip install noisereduce -q
!apt-get install -y ffmpeg > /dev/null 2>&1
print("ffmpeg + noisereduce установлены")


In [ ]:
# Монтирование Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Импорты и настройка устройства
import os, glob, re, random, warnings, pickle
import numpy as np
import librosa
import librosa.display
import noisereduce as nr
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from collections import defaultdict
from scipy.signal import find_peaks
from scipy.ndimage import uniform_filter1d

warnings.filterwarnings('ignore')
random.seed(42); np.random.seed(42); torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


In [ ]:
# Конфигурация (пути и гиперпараметры)
DATASET_PATH = '/content/drive/MyDrive/key_dataset'
TEST_FILE    = '/content/drive/MyDrive/key_dataset/zbmpzbmp.m4a'
SAVE_PATH    = '/content/drive/MyDrive/keystroke_model.pth'
LABELS_PATH  = '/content/drive/MyDrive/keystroke_labels.pkl'

# Аудио
SR           = 22050
N_MELS       = 128
N_FFT        = 1024
HOP_LENGTH   = 256
FIXED_DUR    = 0.60    # секунд — перекрывает даже длинные нажатия
PRE_PAD_MS   = 35      # мс до начала нажатия
POST_PAD_MS  = 90      # мс после отпускания

# Шумоподавление
NOISE_REDUCE = True
NOISE_PROP   = 0.75

# Разбивка датасета
TEST_PER_KEY = 2       # сэмплов на клавишу в тестовой выборке

# Обучение
BATCH_SIZE     = 32
EPOCHS         = 130
LR             = 1e-3
WEIGHT_DECAY   = 1e-4
AUGMENT_FACTOR = 10    # аугментированных копий на тренировочный сэмпл

# Производные параметры
FIXED_SAMPLES = int(FIXED_DUR * SR)
FIXED_FRAMES  = FIXED_SAMPLES // HOP_LENGTH + 1

print(f"Спектрограмма : ({N_MELS}, {FIXED_FRAMES})")
print("Конфиг загружен")


In [ ]:
# Аудиоутилиты: загрузка, шумоподавление, сегментация, mel-спектрограмма

def load_audio(path, sr=SR):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return y

def apply_noise_reduction(y, sr=SR, prop=NOISE_PROP):
    """Оценивает шумовой фон по тихим участкам и подавляет его."""
    if not NOISE_REDUCE:
        return y
    hop = 256
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    thr = np.percentile(rms, 12)
    quiet_idx = np.where(rms < thr * 1.8)[0]
    if len(quiet_idx) > 15:
        noise_chunks = [y[i*hop:min(len(y),(i+1)*hop)] for i in quiet_idx[:120]]
        noise = np.concatenate(noise_chunks)
        if len(noise) > sr * 0.05:
            try:
                return nr.reduce_noise(y=y, y_noise=noise[:int(sr*0.5)], sr=sr,
                                       prop_decrease=prop, time_constant_s=0.02)
            except Exception:
                pass
    try:
        return nr.reduce_noise(y=y, sr=sr, prop_decrease=prop * 0.7)
    except Exception:
        return y

def compute_onset_env(y, sr=SR, hop=128):
    env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop,
                                        aggregate=np.median, n_mels=64)
    env = uniform_filter1d(env, size=3)
    env /= (env.max() + 1e-9)
    return env

def find_transient_peaks(env, sr=SR, hop=128,
                          min_dist_ms=38, height=0.20, prominence=0.08):
    """Возвращает индексы значимых транзиентных пиков в огибающей."""
    min_d = max(2, int(min_dist_ms * sr / (hop * 1000)))
    peaks, _ = find_peaks(env, height=height, distance=min_d, prominence=prominence)
    return peaks

def pair_peaks(peaks, hop=128, sr=SR, max_hold_sec=0.72):
    """
    Объединяет соседние пики в пары (нажатие, отпускание).
    Если разрыв между пиками превышает max_hold_sec — они не спариваются.
    """
    if len(peaks) == 0:
        return []
    max_gap = int(max_hold_sec * sr)
    samp = peaks * hop
    kp = []
    i = 0
    while i < len(samp):
        if i + 1 < len(samp) and (samp[i+1] - samp[i]) <= max_gap:
            kp.append((samp[i], samp[i+1]))
            i += 2
        else:
            kp.append((samp[i], samp[i]))  # одиночный пик
            i += 1
    return kp

def extract_segment(y, press, release, sr=SR):
    """Вырезает сегмент фиксированной длины с нормализацией амплитуды."""
    pre  = int(PRE_PAD_MS  * sr / 1000)
    post = int(POST_PAD_MS * sr / 1000)
    s = max(0, press - pre)
    e = min(len(y), release + post)
    seg = y[s:e]
    if len(seg) < FIXED_SAMPLES:
        seg = np.pad(seg, (0, FIXED_SAMPLES - len(seg)))
    else:
        seg = seg[:FIXED_SAMPLES]
    peak = np.abs(seg).max()
    if peak > 1e-7:
        seg = seg / peak * 0.95
    return seg.astype(np.float32)

def segment_file(path, sr=SR, verbose=True):
    """Полный пайплайн: аудиофайл -> список сегментов нажатий."""
    name = os.path.basename(path)
    y = load_audio(path, sr)
    y = apply_noise_reduction(y, sr)
    env = compute_onset_env(y, sr)
    peaks = find_transient_peaks(env, sr)
    kp_pairs = pair_peaks(peaks, sr=sr)
    segs = [extract_segment(y, p, r, sr) for p, r in kp_pairs]
    if verbose:
        print(f"  {name}: dur={len(y)/sr:.2f}s  peaks={len(peaks)}  keypresses={len(segs)}")
    return segs, y, env, peaks

def seg_to_mel(seg, sr=SR):
    """Сегмент -> нормализованная mel-спектрограмма (n_mels x FIXED_FRAMES)."""
    mel = librosa.feature.melspectrogram(
        y=seg, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH,
        fmin=60, fmax=sr // 2)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    # Выравниваем ось времени
    if mel_db.shape[1] < FIXED_FRAMES:
        mel_db = np.pad(mel_db, ((0,0),(0, FIXED_FRAMES - mel_db.shape[1])))
    else:
        mel_db = mel_db[:, :FIXED_FRAMES]
    # Нормализация min-max
    lo, hi = mel_db.min(), mel_db.max()
    mel_db = (mel_db - lo) / (hi - lo + 1e-9)
    return mel_db.astype(np.float32)

print("Аудиоутилиты загружены")


In [ ]:
# Калибровка сегментации на zbmpzbmp.m4a
# Ожидается 8 нажатий: z b m p z b m p
print("Калибровка сегментации на zbmpzbmp.m4a...")
segs_calib, y_cal, env_cal, pk_cal = segment_file(TEST_FILE, verbose=True)
print(f"\nОбнаружено {len(segs_calib)} нажатий (ожидается 8)")

hop_calib = 128
fig, axes = plt.subplots(3, 1, figsize=(15, 8))
fig.suptitle("Калибровка сегментации — zbmpzbmp.m4a", fontsize=13, fontweight="bold")

# Осциллограмма
t = np.linspace(0, len(y_cal)/SR, len(y_cal))
axes[0].plot(t, y_cal, color="steelblue", lw=0.5)
axes[0].set_title("Осциллограмма (после шумоподавления)")
axes[0].set_xlabel("Время (с)"); axes[0].set_ylabel("Амплитуда")

# Огибающая + пики
t_env = np.arange(len(env_cal)) * hop_calib / SR
axes[1].plot(t_env, env_cal, color="darkorange", lw=1, label="onset strength")
pk_t = pk_cal * hop_calib / SR
axes[1].scatter(pk_t, env_cal[pk_cal], s=60, c="red", zorder=5, label=f"{len(pk_cal)} пиков")
axes[1].set_title("Огибающая нарастания")
axes[1].set_xlabel("Время (с)"); axes[1].legend()

# Mel-спектрограмма
mel_cal = librosa.feature.melspectrogram(y=y_cal, sr=SR, n_mels=64, hop_length=HOP_LENGTH)
librosa.display.specshow(librosa.power_to_db(mel_cal, ref=np.max),
                         sr=SR, hop_length=HOP_LENGTH,
                         x_axis="time", y_axis="mel", ax=axes[2], cmap="magma")
axes[2].set_title("Mel-спектрограмма")

plt.tight_layout()
plt.savefig("/content/calib_segmentation.png", dpi=100)
plt.show()

# Отображение каждого найденного нажатия
if segs_calib:
    n = min(8, len(segs_calib))
    expected_labels = ["z","b","m","p","z","b","m","p"]
    fig2, ax2 = plt.subplots(2, n, figsize=(2*n+1, 4))
    for i in range(n):
        m = seg_to_mel(segs_calib[i])
        ax2[0][i].imshow(m, aspect="auto", origin="lower", cmap="magma")
        ax2[0][i].set_title(f"KP{i+1}\nexp:'{expected_labels[i]}'", fontsize=8)
        ax2[0][i].axis("off")
        ax2[1][i].plot(segs_calib[i], lw=0.5, color="steelblue")
        ax2[1][i].axis("off")
    fig2.suptitle("Отдельные нажатия — zbmpzbmp.m4a", fontsize=11)
    plt.tight_layout(); plt.show()


In [ ]:
# Построение датасета из тренировочных файлов

def parse_key_files(dataset_path):
    """Группирует .m4a-файлы по имени клавиши. Пропускает zbmpzbmp."""
    key_files = defaultdict(list)
    for f in sorted(glob.glob(os.path.join(dataset_path, "*.m4a"))):
        base = os.path.basename(f)
        if "zbmpzbmp" in base.lower():
            continue
        m = re.match(r"^([a-zA-Z0-9_\-]+?)\d*\.m4a$", base, re.IGNORECASE)
        if m:
            key_files[m.group(1).lower()].append(f)
    return key_files

def build_dataset(dataset_path, test_per_key=TEST_PER_KEY):
    key_files = parse_key_files(dataset_path)
    print(f"Найдено клавиш ({len(key_files)}): {sorted(key_files.keys())}")

    raw = defaultdict(list)
    for key in sorted(key_files):
        for fp in sorted(key_files[key]):
            segs, *_ = segment_file(fp, verbose=True)
            raw[key].extend(segs)
        print(f"  key='{key}'  всего сегментов: {len(raw[key])}")

    all_keys = sorted(raw.keys())
    label_to_idx = {k: i for i, k in enumerate(all_keys)}
    idx_to_label = {i: k for k, i in label_to_idx.items()}

    train_X, train_y, test_X, test_y = [], [], [], []
    for key in all_keys:
        segs = raw[key]
        random.shuffle(segs)
        for s in segs[:test_per_key]:
            test_X.append(seg_to_mel(s)); test_y.append(label_to_idx[key])
        for s in segs[test_per_key:]:
            train_X.append(seg_to_mel(s)); train_y.append(label_to_idx[key])

    train_X = np.array(train_X); train_y = np.array(train_y)
    test_X  = np.array(test_X);  test_y  = np.array(test_y)
    print(f"\ntrain: {train_X.shape}  test: {test_X.shape}  классов: {len(all_keys)}")
    return train_X, train_y, test_X, test_y, label_to_idx, idx_to_label

train_X, train_y, test_X, test_y, label_to_idx, idx_to_label = build_dataset(DATASET_PATH)
n_classes = len(label_to_idx)

with open(LABELS_PATH, "wb") as f:
    pickle.dump({"label_to_idx": label_to_idx, "idx_to_label": idx_to_label}, f)
print(f"Метки сохранены -> {LABELS_PATH}")

# Быстрая визуализация нескольких сэмплов
fig, axes = plt.subplots(2, min(10, n_classes), figsize=(min(10,n_classes)*2, 4))
col = 0
for i, key in enumerate(list(idx_to_label.values())[:min(10, n_classes)]):
    idx_list = np.where(train_y == label_to_idx[key])[0]
    if len(idx_list) == 0:
        continue
    mel = train_X[idx_list[0]]
    axes[0][col].imshow(mel, aspect="auto", origin="lower", cmap="magma")
    axes[0][col].set_title(f"'{key}'", fontsize=9)
    axes[0][col].axis("off")
    axes[1][col].plot(mel.mean(0), lw=0.8, color="steelblue")
    axes[1][col].axis("off")
    col += 1

plt.suptitle("Примеры mel-спектрограмм по клавишам", fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# Аугментация и PyTorch Dataset

def augment_mel(mel):
    """Случайные аугментации в пространстве mel-спектрограммы."""
    aug = mel.copy()

    # Гауссовый шум
    if random.random() < 0.55:
        aug += np.random.randn(*aug.shape).astype(np.float32) * random.uniform(0.005, 0.025)
        aug = np.clip(aug, 0, 1)

    # Циклический сдвиг по времени
    if random.random() < 0.55:
        shift = random.randint(-6, 6)
        aug = np.roll(aug, shift, axis=1)
        if shift > 0:
            aug[:, :shift] = 0
        elif shift < 0:
            aug[:, shift:] = 0

    # Яркость (сдвиг в лог-области)
    if random.random() < 0.50:
        aug = np.clip(aug + random.uniform(-0.12, 0.12), 0, 1)

    # Небольшой сдвиг по частоте
    if random.random() < 0.35:
        sh = random.randint(-4, 4)
        aug = np.roll(aug, sh, axis=0)
        if sh > 0:
            aug[:sh, :] = 0
        elif sh < 0:
            aug[sh:, :] = 0

    # Маска по частоте (SpecAugment)
    if random.random() < 0.50:
        f = random.randint(2, 14)
        f0 = random.randint(0, max(1, N_MELS - f))
        aug[f0:f0+f, :] = 0

    # Маска по времени
    if random.random() < 0.50:
        t = random.randint(1, 8)
        t0 = random.randint(0, max(1, FIXED_FRAMES - t))
        aug[:, t0:t0+t] = 0

    return aug


class KeystrokeDataset(Dataset):
    def __init__(self, X, y, augment=False):
        if augment and AUGMENT_FACTOR > 1:
            Xall = [X]
            yall = [y]
            for _ in range(AUGMENT_FACTOR - 1):
                Xall.append(np.array([augment_mel(x) for x in X]))
                yall.append(y)
            self.X = np.concatenate(Xall, 0)
            self.y = np.concatenate(yall, 0)
            print(f"Датасет после аугментации: {len(X)} -> {len(self.X)} сэмплов")
        else:
            self.X = X
            self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.float32).unsqueeze(0)
        return x, int(self.y[idx])


train_ds = KeystrokeDataset(train_X, train_y, augment=True)
test_ds  = KeystrokeDataset(test_X,  test_y,  augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, drop_last=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=0)

print(f"Батчей train: {len(train_loader)}  |  Батчей test: {len(test_loader)}")


In [ ]:
# Модель KeystrokeNet: ResNet + Squeeze-Excitation

class ConvBnAct(nn.Module):
    def __init__(self, cin, cout, k=3, s=1, p=1, groups=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, k, stride=s, padding=p, groups=groups, bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU())
    def forward(self, x): return self.net(x)


class SEBlock(nn.Module):
    """Канальное внимание Squeeze-and-Excitation."""
    def __init__(self, ch, r=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(ch, max(4, ch//r), bias=False),
            nn.ReLU(),
            nn.Linear(max(4, ch//r), ch, bias=False),
            nn.Sigmoid())
    def forward(self, x):
        w = self.pool(x).flatten(1)
        return x * self.fc(w).view(x.shape[0], -1, 1, 1)


class ResBlockSE(nn.Module):
    """Остаточный блок с SE-вниманием и dropout."""
    def __init__(self, ch, drop=0.10):
        super().__init__()
        self.block = nn.Sequential(
            ConvBnAct(ch, ch), nn.Dropout2d(drop), ConvBnAct(ch, ch))
        self.se   = SEBlock(ch)
        self.act  = nn.GELU()
    def forward(self, x):
        return self.act(x + self.se(self.block(x)))


class KeystrokeNet(nn.Module):
    """
    Mel-спектрограмма -> класс клавиши.
    Вход : (B, 1, N_MELS, FIXED_FRAMES)
    Выход: (B, n_classes)
    """
    def __init__(self, n_classes, drop=0.40):
        super().__init__()

        # Stem: (1,128,T) -> (32,64,T/2)
        self.stem = nn.Sequential(
            ConvBnAct(1,  32, k=3, p=1),
            ConvBnAct(32, 32, k=3, p=1),
            nn.MaxPool2d(2, 2))

        # Stage 1: -> (64, 32, T/4)
        self.s1 = nn.Sequential(
            ConvBnAct(32, 64),
            ResBlockSE(64, drop=0.08),
            ResBlockSE(64, drop=0.08),
            nn.MaxPool2d(2, 2))

        # Stage 2: -> (128, 16, T/8)
        self.s2 = nn.Sequential(
            ConvBnAct(64, 128),
            ResBlockSE(128, drop=0.12),
            ResBlockSE(128, drop=0.12),
            nn.MaxPool2d(2, 2))

        # Stage 3: -> (256, 8, T/16)
        self.s3 = nn.Sequential(
            ConvBnAct(128, 256),
            ResBlockSE(256, drop=0.15),
            ResBlockSE(256, drop=0.15),
            nn.AdaptiveAvgPool2d((4, 4)))

        # Голова классификатора
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*4*4, 512, bias=False),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(drop * 0.5),
            nn.Linear(256, n_classes))

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear) and m.weight is not None:
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.head(self.s3(self.s2(self.s1(self.stem(x)))))


model = KeystrokeNet(n_classes).to(device)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Модель    : KeystrokeNet")
print(f"Параметров: {params:,}")
print(f"Классов   : {n_classes}  ->  {list(idx_to_label.values())}")

# Проверка прохождения тензора
dummy = torch.randn(4, 1, N_MELS, FIXED_FRAMES).to(device)
out   = model(dummy)
print(f"Выход     : {tuple(out.shape)}")


In [ ]:
# Обучение

criterion = nn.CrossEntropyLoss(label_smoothing=0.08)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=30, T_mult=2, eta_min=5e-7)

# SpecAugment применяется побатчево на GPU
def spec_augment_batch(x, freq_param=14, time_param=8, nf=2, nt=1):
    x = x.clone()
    B, C, F, T = x.shape
    for _ in range(nf):
        f = random.randint(1, freq_param)
        f0 = random.randint(0, max(1, F - f))
        x[:, :, f0:f0+f, :] = 0
    for _ in range(nt):
        t = random.randint(1, time_param)
        t0 = random.randint(0, max(1, T - t))
        x[:, :, :, t0:t0+t] = 0
    return x

history = {"tl": [], "ta": [], "vl": [], "va": []}
best_acc, best_ep = 0.0, 0

print(f"Обучение {EPOCHS} эпох  |  batch={BATCH_SIZE}  |  lr={LR}")
print("=" * 60)

for ep in range(1, EPOCHS + 1):
    # Тренировка
    model.train()
    tl = tc = tt = 0
    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=True)
        yb = torch.tensor(yb).to(device, non_blocking=True)
        if random.random() < 0.65:
            xb = spec_augment_batch(xb)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tl += loss.item()
        tc += (model(xb).argmax(1) == yb).sum().item()
        tt += len(yb)
    scheduler.step()

    # Валидация
    model.eval()
    vl = vc = vt = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device); yb = torch.tensor(yb).to(device)
            out = model(xb)
            vl += criterion(out, yb).item()
            vc += (out.argmax(1) == yb).sum().item()
            vt += len(yb)

    ta = tc / tt
    va = vc / max(1, vt)
    history["tl"].append(tl / len(train_loader))
    history["ta"].append(ta)
    history["vl"].append(vl / max(1, len(test_loader)))
    history["va"].append(va)

    if va >= best_acc:
        best_acc, best_ep = va, ep
        torch.save({
            "epoch": ep, "model_state": model.state_dict(),
            "val_acc": va, "label_to_idx": label_to_idx,
            "idx_to_label": idx_to_label
        }, SAVE_PATH)

    if ep % 10 == 0 or ep <= 5:
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"Ep {ep:3d}/{EPOCHS}  train={ta:.3f}({history['tl'][-1]:.4f})"
              f"  val={va:.3f}({history['vl'][-1]:.4f})"
              f"  lr={lr_now:.2e}  best={best_acc:.3f}@{best_ep}")

print(f"\nЛучшая val acc: {best_acc*100:.2f}% на эпохе {best_ep}")
print(f"Сохранено -> {SAVE_PATH}")


In [ ]:
# Графики обучения
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("История обучения", fontsize=13, fontweight="bold")
eps = range(1, len(history["tl"]) + 1)

ax1.plot(eps, history["tl"], label="train", color="royalblue")
ax1.plot(eps, history["vl"], label="val",   color="tomato")
ax1.set(xlabel="Эпоха", ylabel="Loss", title="Cross-Entropy Loss")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(eps, [v*100 for v in history["ta"]], label="train", color="royalblue")
ax2.plot(eps, [v*100 for v in history["va"]], label="val",   color="tomato")
ax2.axhline(best_acc*100, ls="--", color="green", alpha=0.7,
            label=f"best val {best_acc*100:.1f}%@ep{best_ep}")
ax2.set(xlabel="Эпоха", ylabel="Точность (%)", title="Точность")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/content/training_history.png", dpi=120)
plt.show()


In [ ]:
# Оценка на тестовой выборке

# Загружаем лучший чекпоинт
ckpt = torch.load(SAVE_PATH)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Загружена модель — эпоха {ckpt['epoch']}  val_acc={ckpt['val_acc']:.4f}")

all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        out = model(xb.to(device))
        all_pred.extend(out.argmax(1).cpu().numpy())
        all_true.extend(yb.numpy() if isinstance(yb, torch.Tensor) else yb)

all_pred = np.array(all_pred); all_true = np.array(all_true)
class_names = [idx_to_label[i] for i in range(n_classes)]
acc = accuracy_score(all_true, all_pred)

print(f"\nТочность на тесте: {acc*100:.2f}%\n")
print(classification_report(all_true, all_pred, target_names=class_names))

cm = confusion_matrix(all_true, all_pred)
fig, ax = plt.subplots(figsize=(max(7, n_classes), max(5, n_classes-2)))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax,
            linewidths=0.5, linecolor="white")
ax.set_xlabel("Предсказание", fontsize=12)
ax.set_ylabel("Истина", fontsize=12)
ax.set_title(f"Матрица ошибок — точность {acc*100:.1f}%", fontsize=13)
plt.tight_layout()
plt.savefig("/content/confusion_matrix.png", dpi=120)
plt.show()


In [ ]:
# Декодирование zbmpzbmp.m4a
# Ожидаемый вывод: z b m p z b m p
EXPECTED = ["z","b","m","p","z","b","m","p"]

def predict_file(path, top_k=3):
    segs, *_ = segment_file(path, verbose=True)
    if not segs:
        print("Нажатия не обнаружены"); return []
    model.eval()
    results = []
    with torch.no_grad():
        for i, seg in enumerate(segs):
            x = torch.tensor(seg_to_mel(seg)).unsqueeze(0).unsqueeze(0).to(device)
            probs = F.softmax(model(x), dim=1)[0].cpu().numpy()
            pred  = int(probs.argmax())
            top_k_list = [(idx_to_label[j], float(probs[j]))
                          for j in probs.argsort()[::-1][:top_k]]
            results.append({"idx": i+1, "label": idx_to_label[pred],
                             "conf": float(probs[pred]), "top": top_k_list})
    return results

print("=" * 50)
results = predict_file(TEST_FILE)
print("\n" + "=" * 50)
if results:
    decoded = " ".join(r["label"] for r in results)
    print(f"\nДекодировано: {decoded}")
    print(f"Ожидалось   : {' '.join(EXPECTED)}")
    if len(results) == len(EXPECTED):
        correct = sum(r["label"] == e for r, e in zip(results, EXPECTED))
        print(f"Точность    : {correct}/{len(EXPECTED)} = {correct/len(EXPECTED)*100:.1f}%")
    print()
    for r in results:
        bar = " | ".join(f"{k}:{v*100:.0f}%" for k,v in r["top"])
        exp = EXPECTED[r["idx"]-1] if r["idx"]-1 < len(EXPECTED) else "?"
        ok  = "OK" if r["label"] == exp else "!!"
        print(f"  KP{r['idx']}: '{r['label']}' ({r['conf']*100:.1f}%)  {ok} exp='{exp}'  [{bar}]")

# Визуализация результатов
if results:
    segs_viz, *_ = segment_file(TEST_FILE, verbose=False)
    n = min(len(results), len(segs_viz))
    fig, axes = plt.subplots(2, n, figsize=(2*n+1, 4))
    for i in range(n):
        r   = results[i]
        exp = EXPECTED[i] if i < len(EXPECTED) else "?"
        ok  = r["label"] == exp
        col = "green" if ok else "crimson"
        mel = seg_to_mel(segs_viz[i])
        axes[0][i].imshow(mel, aspect="auto", origin="lower", cmap="magma")
        axes[0][i].set_title(f"KP{i+1}\n\'{r['label']}\'({r['conf']*100:.0f}%)",
                              fontsize=8, color=col, fontweight="bold")
        axes[0][i].axis("off")
        axes[1][i].plot(segs_viz[i], lw=0.5, color="steelblue")
        axes[1][i].set_title(f"exp:\'{exp}\'", fontsize=7)
        axes[1][i].axis("off")
    fig.suptitle("Предсказания — zbmpzbmp.m4a", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig("/content/predictions_zbmpzbmp.png", dpi=120)
    plt.show()


In [ ]:
# Демо с микрофона браузера
from IPython.display import HTML, display
from google.colab import output
import base64

def start_mic_demo(record_seconds=5):
    """Запускает запись с микрофона и классифицирует нажатия."""
    print(f"Запись {record_seconds}с — печатайте на клавиатуре во время записи!")

    def on_audio(b64_data):
        raw = base64.b64decode(b64_data.split(",")[1])
        with open("/tmp/mic_rec.webm", "wb") as f:
            f.write(raw)
        print("\n--- Результат ---")
        res = predict_file("/tmp/mic_rec.webm")
        if res:
            print("Декодировано:", " ".join(r["label"] for r in res))

    output.register_callback("notebook.on_audio", on_audio)

    html = f"""
    <button onclick="recordNow()" style="padding:8px 16px;font-size:14px;
        background:#4CAF50;color:white;border:none;border-radius:4px;cursor:pointer">
      Начать запись ({record_seconds}с)
    </button>
    <span id="st" style="margin-left:12px;font-size:13px;"></span>
    <script>
    async function recordNow() {{
        document.getElementById("st").innerText = "Запись...";
        const stream = await navigator.mediaDevices.getUserMedia({{audio:true}});
        const rec = new MediaRecorder(stream);
        const chunks = [];
        rec.ondataavailable = e => chunks.push(e.data);
        rec.start();
        await new Promise(r => setTimeout(r, {record_seconds*1000}));
        rec.stop(); stream.getTracks().forEach(t=>t.stop());
        await new Promise(r => rec.onstop = r);
        const blob = new Blob(chunks, {{type:"audio/webm"}});
        const fr = new FileReader();
        fr.readAsDataURL(blob);
        await new Promise(r => fr.onload = r);
        document.getElementById("st").innerText = "Обработка...";
        await google.colab.kernel.invokeFunction("notebook.on_audio",[fr.result],{{}});
        document.getElementById("st").innerText = "Готово";
    }}
    </script>
    """
    display(HTML(html))

start_mic_demo(record_seconds=5)


## Итог и возможные улучшения

| Шаг | Описание |
|-----|----------|
| **Сегментация** | Огибающая onset strength -> попарное сопоставление пиков (нажатие + отпускание) |
| **Признаки** | Нормализованная mel-спектрограмма (128 x ~52 кадра) |
| **Модель** | `KeystrokeNet` — ResNet-блоки + Squeeze-Excitation |
| **Аугментация** | Шум, временной сдвиг, частотный сдвиг, яркость, SpecAugment (x10) |
| **Обучение** | AdamW + Cosine Annealing Warm Restarts, label smoothing |

### Как улучшить точность
- Больше данных: записывать каждую клавишу 5-10 раз вместо 2.
- Разные условия: скорость набора, расстояние до микрофона, уровень шума.
- Ансамбль: 3 модели с разными seed + голосование большинством.
- Подбор порога: подстроить параметр `height` в `find_transient_peaks`, если сегментация пропускает или дробит нажатия.

> Только в образовательных и исследовательских целях.
